In [17]:
import os
import argparse
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon, friedmanchisquare

In [18]:
def holm_correct(p_values):
    """
    Apply Holm-Bonferroni correction to a list of p-values.
    Returns corrected p-values.
    """
    n = len(p_values)
    if n == 0:
        return []
    
    # Sort with indices to keep track of original order
    indexed_p = sorted(enumerate(p_values), key=lambda x: x[1])
    corrected = [0.0] * n
    
    current_max = 0.0
    for rank, (orig_idx, p) in enumerate(indexed_p):
        # Holm formula: p_corr = p * (n - rank)
        p_corr = p * (n - rank)
        p_corr = min(1.0, max(p_corr, current_max))
        current_max = p_corr
        corrected[orig_idx] = p_corr
        
    return corrected

def load_metrics(exp_dir, metric_col):
    summary_path = os.path.join(exp_dir, "subject_summary.csv")
    if not os.path.exists(summary_path):
        raise FileNotFoundError(f"subject_summary.csv not found in {exp_dir}. Run calculate_past_subject_metrics.py first.")
    df = pd.read_csv(summary_path)
    if metric_col not in df.columns:
        raise ValueError(f"Metric '{metric_col}' not found in {summary_path}. Available columns: {list(df.columns)}")
    # Sort by subject_name to ensure pairing is aligned
    df = df.sort_values(by="subject_name").reset_index(drop=True)
    return df

In [19]:
def run_wilcoxon(dir1, dir2, label1, label2, metric):
    df1 = load_metrics(dir1, metric)
    df2 = load_metrics(dir2, metric)
    
    # Check subjects matching
    if not df1["subject_name"].equals(df2["subject_name"]):
        raise ValueError("Subject lists do not match between the two experiments.")
        
    subjects = df1["subject_name"].values
    vals1 = df1[metric].values
    vals2 = df2[metric].values
    
    # Run Wilcoxon signed-rank test
    stat, p_val = wilcoxon(vals1, vals2)
    
    print("\n" + "="*60)
    print(f" Wilcoxon Signed-Rank Test ({metric.upper()})")
    print(f" Group 1 ({label1}) vs Group 2 ({label2})")
    print("="*60)
    
    # Table of values
    df_compare = pd.DataFrame({
        "Subject": subjects,
        label1: vals1,
        label2: vals2,
        "Diff (G2 - G1)": vals2 - vals1
    })
    print(df_compare.to_string(index=False))
    print("-"*60)
    print(f"Mean ({label1}): {np.mean(vals1):.4f} ± {np.std(vals1):.4f}")
    print(f"Mean ({label2}): {np.mean(vals2):.4f} ± {np.std(vals2):.4f}")
    print(f"Wilcoxon Statistic: {stat:.1f}")
    print(f"p-value: {p_val:.6f}")
    if p_val < 0.05:
        print("Result: Significant difference (p < 0.05)")
    else:
        print("Result: No significant difference (p >= 0.05)")
    print("="*60 + "\n")

In [20]:
def run_friedman(dirs, labels, metric):
    if len(dirs) < 3:
        raise ValueError("Friedman test requires at least 3 groups.")
    if len(dirs) != len(labels):
        raise ValueError("Number of directories must match the number of labels.")
        
    dfs = []
    for d in dirs:
        dfs.append(load_metrics(d, metric))
        
    # Check subject names match
    subjects = dfs[0]["subject_name"].values
    for idx, df in enumerate(dfs[1:]):
        if not dfs[0]["subject_name"].equals(df["subject_name"]):
            raise ValueError(f"Subject lists do not match between {labels[0]} and {labels[idx+1]}.")
            
    # Prepare data for Friedman: list of arrays, each array is a group
    group_data = [df[metric].values for df in dfs]
    
    # Run Friedman test
    stat, p_val = friedmanchisquare(*group_data)
    
    print("\n" + "="*80)
    print(f" Friedman Test ({metric.upper()})")
    print(" Groups:", ", ".join(labels))
    print("="*80)
    
    df_compare = pd.DataFrame({"Subject": subjects})
    for lbl, vals in zip(labels, group_data):
        df_compare[lbl] = vals
    print(df_compare.to_string(index=False))
    print("-"*80)
    for lbl, vals in zip(labels, group_data):
        print(f"Mean ({lbl}): {np.mean(vals):.4f} ± {np.std(vals):.4f}")
    print(f"Friedman Chi-Square Statistic: {stat:.4f}")
    print(f"p-value: {p_val:.6f}")
    
    if p_val < 0.05:
        print("Result: Significant difference exists among groups (p < 0.05). Proceeding to Post-hoc pairwise Wilcoxon tests with Holm correction...")
        print("="*80)
        
        # Post-hoc Pairwise Wilcoxon Tests
        pairs = []
        raw_p_values = []
        num_groups = len(labels)
        
        for i in range(num_groups):
            for j in range(i+1, num_groups):
                lbl1, lbl2 = labels[i], labels[j]
                vals1, vals2 = group_data[i], group_data[j]
                _, p = wilcoxon(vals1, vals2)
                pairs.append((lbl1, lbl2, np.mean(vals1), np.mean(vals2)))
                raw_p_values.append(p)
                
        corrected_p_values = holm_correct(raw_p_values)
        
        print(f"\nPost-hoc Pairwise Comparison (Wilcoxon Signed-Rank with Holm correction):")
        print(f"{'Comparison':<35} | {'Mean 1':<8} | {'Mean 2':<8} | {'Raw p-value':<12} | {'Holm p-value':<12} | {'Significant':<12}")
        print("-" * 97)
        for (lbl1, lbl2, m1, m2), raw_p, corr_p in zip(pairs, raw_p_values, corrected_p_values):
            sig = "Yes" if corr_p < 0.05 else "No"
            comp_name = f"{lbl1} vs {lbl2}"
            print(f"{comp_name:<35} | {m1:<8.4f} | {m2:<8.4f} | {raw_p:<12.6f} | {corr_p:<12.6f} | {sig:<12}")
        print("="*80 + "\n")
    else:
        print("Result: No significant difference exists among groups (p >= 0.05). Post-hoc tests skipped.")
        print("="*80 + "\n")

In [21]:
run_friedman(
    dirs=[
        "../outputs/experiments/cnn_grf_single_light_cnn_20260610_042904",
        "../outputs/experiments/bilstm_grf_single_light_bilstm_20260610_043359",
        "../outputs/experiments/transformer_grf_single_transformer_20260421_053321"
    ],
    labels=[
        "CNN",
        "BiLSTM",
        "Transformer"
    ],
    metric="nrmse"
)


 Friedman Test (NRMSE)
 Groups: CNN, BiLSTM, Transformer
 Subject    CNN  BiLSTM  Transformer
  adachi 0.0331  0.0160       0.0171
fukuzawa 0.0365  0.0309       0.0229
 iwasaki 0.0443  0.0260       0.0292
    john 0.0532  0.0362       0.0345
  kiuchi 0.0365  0.0328       0.0251
   konan 0.0342  0.0186       0.0182
    kuno 0.0439  0.0257       0.0236
     oba 0.0477  0.0386       0.0333
   obara 0.0412  0.0289       0.0241
     ono 0.0272  0.0192       0.0187
     pon 0.0431  0.0248       0.0261
  yanaze 0.0348  0.0292       0.0252
--------------------------------------------------------------------------------
Mean (CNN): 0.0396 ± 0.0069
Mean (BiLSTM): 0.0272 ± 0.0067
Mean (Transformer): 0.0248 ± 0.0053
Friedman Chi-Square Statistic: 19.5000
p-value: 0.000058
Result: Significant difference exists among groups (p < 0.05). Proceeding to Post-hoc pairwise Wilcoxon tests with Holm correction...

Post-hoc Pairwise Comparison (Wilcoxon Signed-Rank with Holm correction):
Comparison         

In [26]:
run_friedman(
    dirs=[
        "../outputs/experiments/transformer_grf_single_transformer_20260421_053321",
        "../outputs/experiments/transformer_grf_single_weighted_transformer_20260421_055208",
        "../outputs/experiments/transformer_grf_single_weighted_var_transformer_20260611_042543",
        "../outputs/experiments/transformer_grf_single_weighted_std_transformer_20260611_040709"
    ],
    labels=[
        "None",
        "Fixed",
        "Variance",
        "Std"
    ],
    metric="nrmse"
)


 Friedman Test (NRMSE)
 Groups: None, Fixed, Variance, Std
 Subject   None  Fixed  Variance    Std
  adachi 0.0171 0.0163    0.0252 0.0170
fukuzawa 0.0229 0.0255    0.0337 0.0292
 iwasaki 0.0292 0.0334    0.0608 0.0316
    john 0.0345 0.0429    0.0495 0.0403
  kiuchi 0.0251 0.0272    0.0344 0.0278
   konan 0.0182 0.0224    0.0242 0.0215
    kuno 0.0236 0.0305    0.0483 0.0308
     oba 0.0333 0.0312    0.0412 0.0319
   obara 0.0241 0.0225    0.0260 0.0302
     ono 0.0187 0.0242    0.0319 0.0239
     pon 0.0261 0.0274    0.0461 0.0325
  yanaze 0.0252 0.0306    0.0356 0.0285
--------------------------------------------------------------------------------
Mean (None): 0.0248 ± 0.0053
Mean (Fixed): 0.0278 ± 0.0064
Mean (Variance): 0.0381 ± 0.0108
Mean (Std): 0.0288 ± 0.0057
Friedman Chi-Square Statistic: 23.7000
p-value: 0.000029
Result: Significant difference exists among groups (p < 0.05). Proceeding to Post-hoc pairwise Wilcoxon tests with Holm correction...

Post-hoc Pairwise Compariso

In [27]:
run_wilcoxon(
    dir1="../outputs/experiments/transformer_grf_single_weighted_transformer_20260421_055208",
    dir2="../outputs/experiments/transformer_grf_pressure_single_weighted_transformer_20260421_074422",
    label1="IMU On",
    label2="IMU Off",
    metric="nrmse"
)


 Wilcoxon Signed-Rank Test (NRMSE)
 Group 1 (IMU On) vs Group 2 (IMU Off)
 Subject  IMU On  IMU Off  Diff (G2 - G1)
  adachi  0.0163   0.0179          0.0016
fukuzawa  0.0255   0.0220         -0.0035
 iwasaki  0.0334   0.0237         -0.0097
    john  0.0429   0.0404         -0.0025
  kiuchi  0.0272   0.0229         -0.0043
   konan  0.0224   0.0302          0.0078
    kuno  0.0305   0.0266         -0.0039
     oba  0.0312   0.0337          0.0025
   obara  0.0225   0.0239          0.0014
     ono  0.0242   0.0194         -0.0048
     pon  0.0274   0.0299          0.0025
  yanaze  0.0306   0.0296         -0.0010
------------------------------------------------------------
Mean (IMU On): 0.0278 ± 0.0064
Mean (IMU Off): 0.0267 ± 0.0061
Wilcoxon Statistic: 25.5
p-value: 0.339355
Result: No significant difference (p >= 0.05)



In [28]:
run_wilcoxon(
    dir1="../outputs/experiments/transformer_grf_single_standardized_transformer_20260611_071713",
    dir2="../outputs/experiments/transformer_grf_single_transformer_20260421_053321",
    label1="Z-score On",
    label2="Z-score Off",
    metric="nrmse"
)


 Wilcoxon Signed-Rank Test (NRMSE)
 Group 1 (Z-score On) vs Group 2 (Z-score Off)
 Subject  Z-score On  Z-score Off  Diff (G2 - G1)
  adachi      0.0242       0.0171         -0.0071
fukuzawa      0.0319       0.0229         -0.0090
 iwasaki      0.0568       0.0292         -0.0276
    john      0.0464       0.0345         -0.0119
  kiuchi      0.0336       0.0251         -0.0085
   konan      0.0264       0.0182         -0.0082
    kuno      0.0427       0.0236         -0.0191
     oba      0.0358       0.0333         -0.0025
   obara      0.0299       0.0241         -0.0058
     ono      0.0304       0.0187         -0.0117
     pon      0.0414       0.0261         -0.0153
  yanaze      0.0359       0.0252         -0.0107
------------------------------------------------------------
Mean (Z-score On): 0.0363 ± 0.0088
Mean (Z-score Off): 0.0248 ± 0.0053
Wilcoxon Statistic: 0.0
p-value: 0.000488
Result: Significant difference (p < 0.05)

